## 1. Mediu Colab și configurare

In [ ]:
import importlib.util, subprocess, sys, os, json, shutil
from pathlib import Path

packages = {'torch':'torch','numpy':'numpy','pandas':'pandas','sklearn':'scikit-learn',
            'matplotlib':'matplotlib','requests':'requests','pytest':'pytest'}
missing = [pip for module,pip in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import torch, pandas as pd
ROOT = Path('/content/cisco_telemetry_project') if Path('/content').exists() else Path.cwd()/'cisco_telemetry_project'
ROOT.mkdir(parents=True,exist_ok=True)
os.chdir(ROOT)
Path('src').mkdir(exist_ok=True)
Path('tests').mkdir(exist_ok=True)
sys.path.insert(0,str(ROOT/'src'))
print('Folder:',ROOT, '| Python:',sys.version.split()[0], '| Torch:',torch.__version__,
      '| GPU:',torch.cuda.is_available())

In [ ]:
SCENARIOS = ['0', '2', '4', '9']
LABEL_MODE = 'scenario_proxy'
EVENTS_CSV = None
TIME_COLUMN_OVERRIDES = {}
RESAMPLE_SECONDS = 5
MAX_FEATURES = 16
MAX_FILE_MB = 650
WINDOW = 16
STEP = 8
EMBARGO = 8
BATCH_SIZE = 32
EPOCHS = 3
RUN_TRAINING = True
print('SCENARIOS:',SCENARIOS, '| LABEL_MODE:',LABEL_MODE)

## 2. Cod sursă - integrator Cisco, Dataset, modele, antrenare

In [ ]:
%%writefile src/cisco_integration.py
from __future__ import annotations

import gzip
import io
import json
import math
import re
import zipfile
from pathlib import Path
from urllib.parse import quote

import numpy as np
import pandas as pd
import requests

REPO = "https://api.github.com/repos/cisco-ie/telemetry/contents"
REPO_WEB = "https://github.com/cisco-ie/telemetry"
NORMAL_SCENARIOS = {"0", "1"}
EVENT_SCENARIOS = {"2", "3", "4", "5", "6", "7", "8", "9", "10"}
TIME_NAME = re.compile(r"time|date|epoch|collect|recorded|stamp", re.I)
NON_METRIC = re.compile(r"(^id$|_id$|index|interface|router|device|neighbor|address|prefix|asn|port|timestamp|epoch|time|date|sample|row)", re.I)


def list_scenario_files(scenario: str, session=None) -> list[dict]:
    session = session or requests.Session()
    url = f"{REPO}/{quote(str(scenario))}"
    r = session.get(url, timeout=40, headers={"Accept": "application/vnd.github+json"})
    r.raise_for_status()
    items = r.json()
    if not isinstance(items, list):
        raise ValueError(f"GitHub API nu a returnat lista pentru scenariul {scenario}: {str(items)[:220]}")
    return items


def identify_csv(items: list[dict]) -> dict:
    cand = [it for it in items if it.get("type") == "file" and
            re.search(r"\.csv(?:\.gz|\.zip)?$", it.get("name", ""), re.I)]
    cand = [it for it in cand if not re.search(r"header|definition|case|event|label", it["name"], re.I)]
    if not cand:
        raise ValueError("Nu am identificat fișier .csv/.csv.gz/.csv.zip; verifică listing-ul tipărit.")
    return sorted(cand, key=lambda it: (-int(it.get("size", 0)), it["name"]))[0]


def download_scenarios(scenarios=("0", "2", "4", "9"), directory="data/cisco_raw",
                       max_file_mb=650, session=None) -> dict[str, Path]:
    session = session or requests.Session()
    destination = Path(directory)
    destination.mkdir(parents=True, exist_ok=True)
    downloaded = {}
    manifest = []
    for scenario in map(str, scenarios):
        files = list_scenario_files(scenario, session)
        info = identify_csv(files)
        size = int(info.get("size", 0))
        if size > max_file_mb * 1024 * 1024:
            raise ValueError(f"Scenariul {scenario}: {info['name']} are {size/1048576:.0f} MB, "
                             f"peste MAX_FILE_MB={max_file_mb}. Mărește limita sau selectează un subset.")
        file_out = destination / f"{scenario}_{info['name']}"
        print(f"Cisco {scenario}: {info['name']} ({size / 1048576:.1f} MB)")
        if not file_out.exists() or file_out.stat().st_size != size:
            url = info.get("download_url")
            if not url:
                raise ValueError(f"Fișierul {info['name']} nu are download_url")
            with session.get(url, stream=True, timeout=(25, 150)) as r:
                r.raise_for_status()
                with open(file_out.with_suffix(file_out.suffix + '.tmp'), "wb") as f:
                    for chunk in r.iter_content(1024 * 1024):
                        if chunk:
                            f.write(chunk)
            temp = file_out.with_suffix(file_out.suffix + '.tmp')
            if size and temp.stat().st_size != size:
                raise IOError(f"Fișier incomplet: {temp.stat().st_size} / {size} bytes")
            temp.replace(file_out)
        downloaded[scenario] = file_out
        manifest.append({"scenario": scenario, "file": file_out.name,
                         "size_bytes": size, "source": info.get("html_url")})
    (destination / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    return downloaded


def open_csv(path: str | Path):
    path = Path(path)
    name = path.name.lower()
    if name.endswith(".zip"):
        z = zipfile.ZipFile(path, "r")
        members = [m for m in z.namelist() if m.lower().endswith(".csv") and not m.endswith('/')]
        if not members:
            z.close()
            raise ValueError(f"No CSV inside {path.name}")
        chosen = max(members, key=lambda m: z.getinfo(m).file_size)
        return z, io.TextIOWrapper(z.open(chosen), encoding='utf-8-sig', errors='replace')
    if name.endswith(".gz"):
        return None, gzip.open(path, 'rt', encoding='utf-8-sig', errors='replace')
    return None, open(path, 'r', encoding='utf-8-sig', errors='replace')


def sampled_csv(path: str | Path, rows_per_chunk=300, chunk_size=15000,
                max_sampled_rows=24000) -> pd.DataFrame:
    archive, fp = open_csv(path)
    retained = []
    chunks = 0
    try:
        reader = pd.read_csv(fp, chunksize=chunk_size, low_memory=False,
                             on_bad_lines="skip", dtype=str)
        for chunk in reader:
            if chunk.empty:
                continue
            chunks += 1
            positions = np.linspace(0, len(chunk) - 1, num=min(rows_per_chunk, len(chunk)), dtype=int)
            retained.append(chunk.iloc[positions].copy())
            if sum(len(x) for x in retained) > 2 * max_sampled_rows:
                retained = [pd.concat(retained, ignore_index=True).iloc[::2].reset_index(drop=True)]
    finally:
        fp.close()
        if archive is not None:
            archive.close()
    if not retained:
        raise ValueError(f"No readable rows in {path}")
    df = pd.concat(retained, ignore_index=True)
    if len(df) > max_sampled_rows:
        df = df.iloc[np.linspace(0, len(df) - 1, max_sampled_rows, dtype=int)].reset_index(drop=True)
    print(f" {Path(path).name}: scanned {chunks} chunks; sampled {len(df)} rows; columns={len(df.columns)}")
    return df


def parse_time(values: pd.Series) -> pd.Series:
    s = values.astype('string').str.strip()
    num = pd.to_numeric(s, errors='coerce')
    mostly_numeric = float(num.notna().mean()) > 0.85
    if mostly_numeric:
        median = num.abs().median()
        if 1e9 <= median < 1e11: unit = 's'
        elif 1e11 <= median < 1e14: unit = 'ms'
        elif 1e14 <= median < 1e17: unit = 'us'
        elif 1e17 <= median < 1e20: unit = 'ns'
        else: return pd.Series(pd.NaT, index=s.index, dtype='datetime64[ns, UTC]')
        result = pd.to_datetime(num, unit=unit, utc=True, errors='coerce')
    else:
        result = pd.to_datetime(s, utc=True, errors='coerce', format='mixed')
    return result


def find_timestamp_column(df: pd.DataFrame, override: str | None = None):
    if override:
        if override not in df:
            raise ValueError(f"TIME_COLUMN={override!r} nu există. Coloane: {list(df.columns)[:30]}")
        ts = parse_time(df[override])
        if ts.notna().sum() < 20:
            raise ValueError(f"TIME_COLUMN={override!r} does not parse as date/Unix epoch")
        return override, ts
    candidates = [c for c in df if TIME_NAME.search(str(c))]
    candidates += [c for c in list(df)[:8] if c not in candidates]
    best = None
    for c in candidates[:25]:
        ts = parse_time(df[c]); valid = ts.dropna()
        if len(valid) < 20:
            continue
        plausible = valid[(valid.dt.year >= 2000) & (valid.dt.year <= 2040)]
        uniqueness = plausible.nunique()
        ratio = float(len(plausible)) / len(df)
        if uniqueness < 15 or ratio < 0.75:
            continue
        score = (1 if TIME_NAME.search(str(c)) else 0, ratio, uniqueness)
        if best is None or score > best[0]:
            best = (score, c, ts)
    if best is None:
        raise ValueError("Coloana temporală nu poate fi identificată sigur. "
                         "Setează TIME_COLUMN_OVERRIDES în notebook. "
                         f"Primele coloane sunt: {list(df.columns)[:30]}")
    return best[1], best[2]



FEATURE_PRIORITY = (
    'input-data-rate', 'output-data-rate',
    'input-packet-rate', 'output-packet-rate',
    'input-load', 'output-load',
    'input-drops', 'output-drops', 'input-errors', 'crc-errors',
    'carrier-transitions', 'packets-received', 'packets-sent',
    'bytes-received', 'bytes-sent',
    'reliability', 'total-cpu-one-minute', 'total-cpu-five-minute',
    'free-physical-memory', 'bandwidth',
)


def sanitize_numeric(df: pd.DataFrame, timestamp_col: str, min_row_coverage=0.005) -> pd.DataFrame:
    numeric = {}
    for c in df.columns:
        if c == timestamp_col or NON_METRIC.search(str(c)):
            continue
        v = pd.to_numeric(df[c].astype('string').str.strip(), errors='coerce')
        if v.notna().mean() >= min_row_coverage and v.nunique(dropna=True) >= 2:
            numeric[c] = v.astype(float)
    if not numeric:
        raise ValueError(
            'Nicio metrica numerica variabila in CSV. Verifica coloanele brute; '
            'NU folosi automat bandwidth constant ca feature de model.'
        )
    return pd.DataFrame(numeric, index=df.index)


def load_cisco_timeseries(files: dict[str, Path], resample_seconds=5,
                          override_timestamp: dict | None=None, max_features=16):
    override_timestamp = override_timestamp or {}
    provisional = {}
    for scenario, path in files.items():
        raw = sampled_csv(path)
        timestamp_col, ts = find_timestamp_column(raw, override_timestamp.get(str(scenario)))
        num = sanitize_numeric(raw, timestamp_col)
        num.insert(0, '__timestamp', ts)
        num = num.dropna(subset=['__timestamp'])
        num = num[(num['__timestamp'].dt.year >= 2000) & (num['__timestamp'].dt.year <= 2040)]
        if num.empty:
            raise ValueError(f'Scenariul {scenario}: nu au ramas timestamps valide')
        num['__timestamp'] = num['__timestamp'].dt.floor(f'{int(resample_seconds)}s')
        agg = num.groupby('__timestamp', as_index=False).median(numeric_only=True)
        provisional[str(scenario)] = agg
        print(f' Scenario {scenario}: {len(raw)} randuri esantionate, '
              f'{len(num.columns)-1} metrici numerice candidate, {len(agg)} intervale temporale')

    shared = set.intersection(*(set(x.columns) - {'__timestamp'} for x in provisional.values()))
    if not shared:
        raise ValueError('Nicio metrica comuna: este necesar mapping explicit intre scenarii.')

    diagnostic = []
    selected = []
    for col in shared:
        coverages = [float(frame[col].notna().mean()) for frame in provisional.values()]
        unique_per_scenario = [int(frame[col].nunique(dropna=True)) for frame in provisional.values()]
        variating = sum(v > 1 for v in unique_per_scenario)
        good = min(coverages) >= 0.02 and variating >= 2
        diagnostic.append({'feature':col, 'min_time_coverage':round(min(coverages),3),
                           'varying_scenarios':variating, 'selected_candidate':good})
        if good: selected.append(col)

    def rank(col):
        if col in FEATURE_PRIORITY:
            return (0, FEATURE_PRIORITY.index(col), col)
        return (1, -min(provisional[s][col].notna().mean() for s in provisional), col)

    selected = sorted(selected,key=rank)[:max_features]
    print('Feature quality after resampling (shared schema):')
    for item in sorted(diagnostic,key=lambda e:rank(e['feature'])):
        if item['selected_candidate']:
            print(f"  OK {item['feature']:<35} min time coverage={item['min_time_coverage']:.1%}; "
                  f"varying scenarios={item['varying_scenarios']}")
    if len(selected) < 4:
        raise ValueError(
            f'Au ramas doar {len(selected)} metrici variabile comune: {selected}. '
            'Nu continua antrenarea; inspecteaza setul original/sampling/resampling.'
        )
    print(f'Selected {len(selected)} real features:', selected)
    out = {scene:frame[['__timestamp']+selected].copy()
           for scene,frame in provisional.items()}
    for scene,frame in out.items():
        vary = {c:int(frame[c].nunique(dropna=True)) for c in selected}
        print(f'Scenario {scene} feature variability: ', vary)
    return out,selected


def event_labels(times: pd.Series, scenario: str, events: pd.DataFrame):
    matches = events.loc[events.scenario.astype(str) == str(scenario)]
    labels = np.zeros(len(times), dtype=np.int8)
    for _,ev in matches.iterrows():
        start, end = pd.to_datetime(ev['start'], utc=True), pd.to_datetime(ev['end'], utc=True)
        if end < start: raise ValueError('Invalid event interval end before start')
        labels |= ((times >= start) & (times <= end)).to_numpy().astype(np.int8)
    return labels


def canonicalize_and_split(series: dict[str, pd.DataFrame],
                           output='data/cisco_labeled.csv', mode='scenario_proxy',
                           events_csv=None, window=16, embargo=8,
                           split_ratios=(0.70, 0.15, 0.15)):
    if mode not in ('scenario_proxy','event_intervals'):
        raise ValueError("mode must be scenario_proxy or event_intervals")
    if not np.isclose(sum(split_ratios), 1):
        raise ValueError('split_ratios must sum to 1')
    events = None
    if mode == 'event_intervals':
        if events_csv is None or not Path(events_csv).exists():
            raise FileNotFoundError('events_csv with scenario,start,end intervals required')
        events = pd.read_csv(events_csv)
        if not {'scenario','start','end'}.issubset(events):
            raise ValueError('events CSV must have scenario,start,end columns')
        annotated = set(events.scenario.astype(str))
        missing = {s for s in series if s not in NORMAL_SCENARIOS} - annotated
        if missing:
            raise ValueError(f'No reviewed event intervals for anomaly scenarios: {sorted(missing)}. '
                             'Do not treat absent annotations as normal ground truth.')
    parts = []
    audit = {}
    groups = {'train': [], 'val': [], 'test': []}
    for scenario,frame in series.items():
        frame = frame.sort_values('__timestamp').drop_duplicates('__timestamp').reset_index(drop=True)
        n = len(frame)
        cut1, cut2 = math.floor(n * split_ratios[0]), math.floor(n * sum(split_ratios[:2]))
        ranges = {
            'train': (0, cut1 - embargo),
            'val': (cut1 + embargo, cut2 - embargo),
            'test': (cut2 + embargo, n)
        }
        if any(b-a < window for a,b in ranges.values()):
            raise ValueError(f'Scenario {scenario}: {n} time bins => too few for '
                             f'window={window}, embargo={embargo}, ratios={split_ratios}. '
                             'Lower resample_seconds/window or add more data.')
        if mode == 'scenario_proxy':
            if scenario not in NORMAL_SCENARIOS | EVENT_SCENARIOS:
                raise ValueError(f'Unknown scenario: {scenario}')
            scene_labels = np.full(n, 0 if scenario in NORMAL_SCENARIOS else 1, dtype=np.int8)
        else:
            scene_labels = event_labels(frame['__timestamp'], scenario, events)
        audit[scenario] = {}
        for name,(a,b) in ranges.items():
            x = frame.iloc[a:b].copy()
            scene_key = f'{scenario}__{name}'
            x.insert(0, 'scenario', scene_key)
            x.insert(1, 'timestamp', x.pop('__timestamp').astype(str))
            x.insert(2, 'label', scene_labels[a:b])
            parts.append(x)
            groups[name].append(scene_key)
            audit[scenario][name] = {'rows':len(x),'normal':int((scene_labels[a:b] == 0).sum()),
                                     'anomalous':int((scene_labels[a:b] == 1).sum()),
                                     'start':str(frame.iloc[a]['__timestamp']),
                                     'end':str(frame.iloc[b-1]['__timestamp'])}
    df = pd.concat(parts,ignore_index=True)
    out = Path(output); out.parent.mkdir(parents=True,exist_ok=True)
    df.to_csv(out,index=False)
    (out.parent/'split_audit.json').write_text(json.dumps({'label_mode':mode,'audit':audit,'groups':groups},indent=2),encoding='utf-8')
    print(f'Canonical CSV {out} rows={len(df)} features={len(df.columns)-3}, label_mode={mode}')
    return out, groups, audit


def download_case_images(scenarios=("0", "2", "4", "9"), directory="data/cisco_case_files",
                         session=None, max_size_mb=20):
    session = session or requests.Session()
    destination = Path(directory)
    destination.mkdir(parents=True, exist_ok=True)
    results = {}
    for scenario in map(str, scenarios):
        items = list_scenario_files(scenario, session=session)
        candidates = [item for item in items if re.search(r'\.(?:png|jpe?g)$', item.get('name',''),re.I)
                      and re.search(r'case|event|key|anomal', item.get('name',''),re.I)]
        for item in candidates:
            if int(item.get('size',0)) > max_size_mb*1024*1024:
                continue
            url = item.get('download_url')
            if not url:
                continue
            path = destination / f'{scenario}_{item["name"]}'
            if not path.exists() or path.stat().st_size != int(item.get('size',0)):
                r = session.get(url,timeout=75)
                r.raise_for_status()
                path.write_bytes(r.content)
            results.setdefault(scenario,[]).append(path)
    return results


In [ ]:
%%writefile src/data_pipeline.py
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset


class TelemetryWindowDataset(Dataset):

    def __init__(self, images: np.ndarray, labels: np.ndarray, scenario_ids: list[str]):
        assert images.ndim == 4 and images.shape[1] == 1
        assert len(images) == len(labels) == len(scenario_ids)
        self.images = torch.from_numpy(np.asarray(images, dtype=np.float32))
        self.labels = torch.from_numpy(np.asarray(labels, dtype=np.float32))
        self.scenario_ids = scenario_ids

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, index: int):
        return self.images[index], self.labels[index]


@dataclass
class DataBundle:
    loaders: dict[str, DataLoader]
    datasets: dict[str, TelemetryWindowDataset]
    feature_names: list[str]
    split_groups: dict[str, list[str]]
    summary: dict[str, dict[str, int]]


def prepare_data(
    csv_path: str | Path,
    splits: dict[str, list[str]],
    window: int = 32,
    step: int = 16,
    batch_size: int = 32,
    seed: int = 42,
) -> DataBundle:
    if window < 4 or window % 4 != 0 or step <= 0:
        raise ValueError("window must be a multiple of 4 (>=4); step must be positive")
    if set(splits) != {"train", "val", "test"}:
        raise ValueError("splits must have exactly train, val, test")
    groups = {k: [str(x) for x in v] for k, v in splits.items()}
    all_groups = [g for ids in groups.values() for g in ids]
    if len(all_groups) != len(set(all_groups)):
        raise ValueError("Split group IDs must be disjoint across train/val/test")

    df = pd.read_csv(csv_path, low_memory=False)
    needed = {"scenario", "timestamp", "label"}
    if missing := (needed - set(df.columns)):
        raise ValueError(f"Missing canonical CSV columns: {sorted(missing)}")
    df["scenario"] = df["scenario"].astype(str)
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
    if df["timestamp"].isna().any():
        raise ValueError("Invalid timestamp: inspect source CSV and timestamp format")
    df["label"] = pd.to_numeric(df["label"], errors="raise")
    if not df["label"].isin([0, 1]).all():
        raise ValueError("Only verified binary labels (0,1) are supported")
    known_groups = set(df["scenario"].unique())
    if unused := set(all_groups) - known_groups:
        raise ValueError(f"Requested scenario(s) do not exist: {sorted(unused)}")
    if extra := known_groups - set(all_groups):
        raise ValueError(f"Assign ALL scenarios to a split or filter CSV first: {sorted(extra)}")
    feature_cols = [c for c in df.columns if c not in {"scenario", "timestamp", "label"}]
    if not feature_cols:
        raise ValueError("No numeric telemetry features found")
    for name in feature_cols:
        df[name] = pd.to_numeric(df[name], errors="coerce")
    df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan)

    train = df[df.scenario.isin(groups["train"])]
    if train.empty:
        raise ValueError("Train split is empty")
    before = list(feature_cols)
    feature_cols = [c for c in feature_cols
                    if train[c].notna().sum() >= 5
                    and train[c].nunique(dropna=True) > 1]
    removed = sorted(set(before) - set(feature_cols))
    if removed:
        print('Dropped empty/constant features based on TRAIN only:', removed)
    if len(feature_cols) < 4:
        raise ValueError(f'Train has only {len(feature_cols)} informative features: '
                         f'{feature_cols}. Check telemetry aggregation.')
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    x_train = imputer.fit_transform(train[feature_cols])
    scaler.fit(x_train)
    padded_features = int(np.ceil(len(feature_cols) / 4) * 4)

    datasets = {}
    summary = {}
    for part in ("train", "val", "test"):
        subset = df[df.scenario.isin(groups[part])].copy()
        if subset.empty:
            raise ValueError(f"{part} split is empty")
        xs = scaler.transform(imputer.transform(subset[feature_cols]))
        xs = np.clip(xs, -5.0, 5.0).astype(np.float32)
        subset["__row_number"] = np.arange(len(subset))
        images, y, ids = [], [], []
        for scenario_id, scene in subset.groupby("scenario", sort=False):
            scene = scene.sort_values("timestamp", kind="stable")
            indexes = scene["__row_number"].to_numpy()
            labels = scene["label"].to_numpy(dtype=np.int64)
            if len(indexes) < window:
                raise ValueError(f"Scenario {scenario_id} has fewer than {window} rows")
            for start in range(0, len(indexes) - window + 1, step):
                chunk = xs[indexes[start:start + window]]
                if padded_features != len(feature_cols):
                    chunk = np.pad(chunk, ((0, 0), (0, padded_features - len(feature_cols))))
                images.append(chunk[None, ...])
                y.append(int(labels[start:start + window].any()))
                ids.append(scenario_id)
        if not images:
            raise ValueError(f"No windows in split {part}")
        dataset = TelemetryWindowDataset(np.stack(images), np.asarray(y), ids)
        datasets[part] = dataset
        summary[part] = {
            "windows": len(dataset),
            "normal": int((dataset.labels == 0).sum()),
            "anomalous": int((dataset.labels == 1).sum()),
            "scenarios": len(groups[part]),
        }

    generator = torch.Generator().manual_seed(seed)
    loaders = {
        part: DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=(part == "train"),
            num_workers=0,
            generator=generator if part == "train" else None,
        )
        for part, dataset in datasets.items()
    }
    return DataBundle(loaders, datasets, feature_cols, groups, summary)


In [ ]:
%%writefile src/models.py
import torch
from torch import nn


class CNNClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((2, 2)), nn.Flatten(),
            nn.Linear(16 * 2 * 2, 16), nn.ReLU(), nn.Linear(16, 1),
        )

    def forward(self, x: torch.Tensor):
        return self.network(x).squeeze(1)


class ConvAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, stride=2, padding=1), nn.ReLU(),
            nn.Conv2d(8, 16, kernel_size=3, stride=2, padding=1), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(16, 8, kernel_size=4, stride=2, padding=1), nn.ReLU(),
            nn.ConvTranspose2d(8, 1, kernel_size=4, stride=2, padding=1),
        )

    def forward(self, x: torch.Tensor):
        return self.decoder(self.encoder(x))


In [ ]:
%%writefile src/train_compare.py
from __future__ import annotations

import argparse
import copy
import json
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from torch import nn
from torch.utils.data import DataLoader, Subset

from data_pipeline import prepare_data
from models import CNNClassifier, ConvAutoencoder


def parse_ids(value: str) -> list[str]:
    return [v.strip() for v in value.split(",") if v.strip()]


def classification_metrics(truth: np.ndarray, pred: np.ndarray) -> dict:
    return {
        "precision": float(precision_score(truth, pred, zero_division=0)),
        "recall": float(recall_score(truth, pred, zero_division=0)),
        "f1": float(f1_score(truth, pred, zero_division=0)),
        "accuracy": float(accuracy_score(truth, pred)),
        "confusion_matrix_0_1": confusion_matrix(truth, pred, labels=[0, 1]).tolist(),
    }


@torch.no_grad()
def cnn_predict(model: nn.Module, loader: DataLoader, device: torch.device):
    model.eval()
    scores, labels = [], []
    for x, y in loader:
        z = model(x.to(device))
        scores.extend(torch.sigmoid(z).cpu().numpy().tolist())
        labels.extend(y.numpy().astype(int).tolist())
    return np.asarray(labels), np.asarray(scores)


@torch.no_grad()
def ae_predict(model: nn.Module, loader: DataLoader, device: torch.device):
    model.eval()
    scores, labels = [], []
    for x, y in loader:
        x = x.to(device)
        errs = torch.mean((model(x) - x) ** 2, dim=(1, 2, 3))
        scores.extend(errs.cpu().numpy().tolist())
        labels.extend(y.numpy().astype(int).tolist())
    return np.asarray(labels), np.asarray(scores)


def select_val_threshold(labels: np.ndarray, errors: np.ndarray) -> tuple[float, float]:
    if len(np.unique(labels)) != 2:
        raise ValueError("AE threshold selection needs BOTH normal and abnormal windows in validation")
    uniq = np.unique(errors)
    mids = (uniq[:-1] + uniq[1:]) / 2.0
    candidates = np.concatenate(([float(uniq[0]) - 1e-8], mids, [float(uniq[-1]) + 1e-8]))
    best = max(((f1_score(labels, errors > t, zero_division=0), float(t)) for t in candidates),
               key=lambda item: (item[0], -item[1]))
    return best[1], float(best[0])


def train_cnn(bundle, device, epochs: int, lr: float):
    model = CNNClassifier().to(device)
    counts = bundle.summary["train"]
    if counts["anomalous"] == 0 or counts["normal"] == 0:
        raise ValueError("Supervised CNN needs normal AND abnormal windows in train")
    pos_weight = torch.tensor([counts["normal"] / counts["anomalous"]], device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_f1, best_epoch, best_weights = -1.0, 0, None
    log = []
    for epoch in range(1, epochs + 1):
        model.train()
        losses = []
        for x, y in bundle.loaders["train"]:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            losses.append(float(loss.item()))
        val_y, val_prob = cnn_predict(model, bundle.loaders["val"], device)
        val_f1 = float(f1_score(val_y, val_prob >= 0.5, zero_division=0))
        log.append({"model": "CNN", "epoch": epoch, "train_loss": float(np.mean(losses)), "val_f1": val_f1})
        if val_f1 > best_f1:
            best_f1, best_epoch, best_weights = val_f1, epoch, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_weights)
    return model, log, {"selected_epoch": best_epoch, "val_f1": best_f1, "threshold": 0.5}


def train_ae(bundle, device, epochs: int, lr: float, batch_size: int):
    indices = torch.where(bundle.datasets["train"].labels == 0)[0].tolist()
    if not indices:
        raise ValueError("No clean/normal windows available for AE train")
    loader = DataLoader(Subset(bundle.datasets["train"], indices), batch_size=batch_size, shuffle=True)
    model = ConvAutoencoder().to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    best_loss = float("inf")
    best_state, best_epoch = None, 0
    log = []
    for epoch in range(1, epochs + 1):
        model.train()
        losses = []
        for x, _ in loader:
            x = x.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), x)
            loss.backward()
            optimizer.step()
            losses.append(float(loss.item()))
        val_y, val_errors = ae_predict(model, bundle.loaders["val"], device)
        clean = val_errors[val_y == 0]
        if not len(clean):
            raise ValueError("Autoencoder needs normal windows in validation")
        val_loss = float(clean.mean())
        log.append({"model": "Autoencoder", "epoch": epoch,
                    "train_loss": float(np.mean(losses)), "val_normal_mse": val_loss})
        if val_loss < best_loss:
            best_loss, best_epoch, best_state = val_loss, epoch, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    val_y, val_err = ae_predict(model, bundle.loaders["val"], device)
    threshold, val_f1 = select_val_threshold(val_y, val_err)
    return model, log, {"selected_epoch": best_epoch, "val_normal_mse": best_loss,
                        "val_f1_at_selected_threshold": val_f1, "threshold": threshold}


def make_plots(metrics: dict, logs: list[dict], out_dir: Path):
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.4))
    for ax, model in zip(axes, ["CNN", "Autoencoder"]):
        m = metrics[model]["test"]
        cm = np.array(m["confusion_matrix_0_1"])
        ax.imshow(cm, cmap="Blues")
        for (i, j), val in np.ndenumerate(cm):
            ax.text(j, i, str(val), ha="center", va="center")
        ax.set(xlabel="Prediction", ylabel="True label", title=model,
               xticks=[0, 1], yticks=[0, 1])
    fig.tight_layout()
    fig.savefig(out_dir / "confusion_matrices.png", dpi=150)
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(7, 3.4))
    for model in ["CNN", "Autoencoder"]:
        recs = [r for r in logs if r["model"] == model]
        ax.plot([r["epoch"] for r in recs], [r["train_loss"] for r in recs], label=model)
    ax.set(xlabel="Epoch", ylabel="Training loss (different loss functions)", title="Training progression")
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_dir / "loss_curves.png", dpi=150)
    plt.close(fig)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True, help="Canonical labeled CSV, NOT unmodified raw Cisco ZIP")
    ap.add_argument("--train", default="0,2")
    ap.add_argument("--val", default="4")
    ap.add_argument("--test", default="9")
    ap.add_argument("--window", type=int, default=32)
    ap.add_argument("--step", type=int, default=16)
    ap.add_argument("--batch-size", type=int, default=32)
    ap.add_argument("--epochs", type=int, default=12)
    ap.add_argument("--lr", type=float, default=0.001)
    ap.add_argument("--output", default="results")
    args = ap.parse_args()
    torch.manual_seed(42)
    np.random.seed(42)
    torch.set_num_threads(min(torch.get_num_threads(), 2))
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    splits = {"train": parse_ids(args.train), "val": parse_ids(args.val), "test": parse_ids(args.test)}
    bundle = prepare_data(args.data, splits, args.window, args.step, args.batch_size)
    print("Split audit (partition groups disjoint; chronological scenario-level embargo is upstream):", json.dumps(bundle.summary, indent=2))
    x, y = next(iter(bundle.loaders["train"]))
    print(f"DataLoader batch: X={tuple(x.shape)}, y={tuple(y.shape)}; device={device}")
    output = Path(args.output)
    output.mkdir(parents=True, exist_ok=True)
    cnn, log_cnn, cnn_info = train_cnn(bundle, device, args.epochs, args.lr)
    ae, log_ae, ae_info = train_ae(bundle, device, args.epochs, args.lr, args.batch_size)
    test_y, cnn_scores = cnn_predict(cnn, bundle.loaders["test"], device)
    test_y_ae, ae_scores = ae_predict(ae, bundle.loaders["test"], device)
    assert np.array_equal(test_y, test_y_ae), "Same test set is required for fair comparison"
    if len(np.unique(test_y)) < 2:
        print("WARNING: Test set has only one class: F1 is not a meaningful comparison")
    metrics = {
        "CNN": {"selection": cnn_info, "test": classification_metrics(test_y, (cnn_scores >= 0.5).astype(int))},
        "Autoencoder": {"selection": ae_info, "test": classification_metrics(test_y, (ae_scores > ae_info["threshold"]).astype(int))},
        "split_audit": bundle.summary,
        "split_groups": bundle.split_groups,
        "features": bundle.feature_names,
        "notes": "AE threshold determined ONLY on validation; test is held out. Event labels must be audited externally. Scenario-proxy labels are NOT event ground truth. Splits may reuse original scenarios in disjoint time blocks.",
    }
    (output / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
    history = log_cnn + log_ae
    pd.DataFrame(history).to_csv(output / "history.csv", index=False)
    make_plots(metrics, history, output)
    torch.save(cnn.state_dict(), output / "cnn.pt")
    torch.save(ae.state_dict(), output / "autoencoder.pt")
    print(json.dumps({k: v["test"] for k, v in metrics.items() if k in ("CNN", "Autoencoder")}, indent=2))
    print(f"Saved metrics, model weights and plots in {output.resolve()}")


if __name__ == "__main__":
    main()


## 3. Teste offline pentru validarea codului

In [ ]:
%%writefile tests/test_cisco_integration.py
from pathlib import Path
import sys
import gzip
import io
import zipfile

import numpy as np
import pandas as pd
import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))
from cisco_integration import (parse_time, sampled_csv, load_cisco_timeseries,
                               canonicalize_and_split, event_labels)
from data_pipeline import prepare_data


def build_local_fixture(tmp_path, n=650):
    paths = {}
    for scene in ['0','2','4','9']:
        rng = np.random.default_rng(42+int(scene))
        frame = pd.DataFrame({
            'timestamp': pd.date_range('2018-07-24',periods=n,freq='5s',tz='UTC').astype(str),
            'cpu_load': rng.normal(int(scene),1,n),
            'interface_errors': rng.poisson(5+int(scene),n),
            'device_id': ['router-1']*n,
            'drop_rate': rng.random(n),
            'error_counter': rng.poisson(5,n),
            'tx_bytes': rng.integers(1,10000,n),
        })
        path = tmp_path / f'{scene}.csv.{"zip" if scene in ["2","9"] else "gz"}'
        if path.suffix=='.zip':
            with zipfile.ZipFile(path,'w',zipfile.ZIP_DEFLATED) as z:
                z.writestr('telemetry.csv',frame.to_csv(index=False))
        else:
            with gzip.open(path,'wt') as f: frame.to_csv(f,index=False)
        paths[scene]=path
    return paths


def test_parse_unix_and_calendar():
    assert parse_time(pd.Series(['1532390400','1532390405'])).notna().all()
    assert parse_time(pd.Series(['2018-07-24T00:00:00Z','2018-07-24T00:00:05Z'])).notna().all()


def test_real_archive_reader_and_preprocessing(tmp_path):
    files=build_local_fixture(tmp_path)
    series,features=load_cisco_timeseries(files,resample_seconds=5,max_features=12)
    assert set(series)=={'0','2','4','9'}
    assert 'cpu_load' in features
    output,splits,audit=canonicalize_and_split(series,output=tmp_path/'canonical.csv',window=16,embargo=8)
    assert output.exists()
    bundle=prepare_data(output,splits=splits,window=16,step=8,batch_size=24)
    for part in ['train','val','test']:
        x,y=next(iter(bundle.loaders[part])); assert x.ndim==4 and x.shape[2]==16
        assert y.ndim==1
        assert bundle.summary[part]['normal']>0
        assert bundle.summary[part]['anomalous']>0
    for s in ['0','2','4','9']:
        assert audit[s]['train']['end']<audit[s]['val']['start']
        assert audit[s]['val']['end']<audit[s]['test']['start']


def test_event_intervals_not_silently_guessed(tmp_path):
    files=build_local_fixture(tmp_path,n=500)
    series,_=load_cisco_timeseries(files)
    with pytest.raises(FileNotFoundError):
        canonicalize_and_split(series,output=tmp_path/'out.csv',mode='event_intervals')
    events=tmp_path/'events.csv'
    pd.DataFrame({'scenario':['2','4'], 'start':['2018-07-24T00:10:00Z']*2,
                  'end':['2018-07-24T00:20:00Z']*2}).to_csv(events,index=False)
    with pytest.raises(ValueError,match='No reviewed event intervals'):
        canonicalize_and_split(series,output=tmp_path/'out.csv',mode='event_intervals',events_csv=events)


def test_sparse_asynchronous_metric_selection(tmp_path):
    scenes = {}
    for scene in ['0','2','4','9']:
        rng = np.random.default_rng(100+int(scene))
        n = 1800
        timestamps = pd.date_range('2018-07-24', periods=600, freq='5s',tz='UTC')
        frame = pd.DataFrame({'time':np.repeat(timestamps.astype(str),3),
                              'bandwidth':100_000_000.0})
        interface = np.arange(n)%3 == 0
        for feature in ['input-data-rate','output-data-rate','input-errors',
                        'output-drops','carrier-transitions','input-packet-rate']:
            values = np.full(n,np.nan)
            values[interface] = rng.uniform(1,100,interface.sum()) + np.arange(interface.sum())*.05
            frame[feature] = values
        file = tmp_path/f'{scene}.csv.gz'
        with gzip.open(file,'wt') as fp:
            frame.to_csv(fp,index=False)
        scenes[scene] = file
    result, cols = load_cisco_timeseries(scenes,resample_seconds=5,max_features=16)
    assert 'bandwidth' not in cols
    assert len(cols)>=4
    assert 'input-data-rate' in cols
    assert result['9']['output-data-rate'].std()>0


In [ ]:
r = subprocess.run([sys.executable,'-m','pytest','-q','tests/test_cisco_integration.py'],
                   capture_output=True,text=True)
print(r.stdout)
if r.stderr: print(r.stderr[-2000:])
assert r.returncode == 0, 'Testele de integrare au eșuat'

## 4a. Descărcare seturilor Cisco de pe GitHub

Cele patru arhive sunt selectate folderele `0`, `2`, `4`, `9`. Se folosește GitHub API, iar fișierele sunt păstrate în cache-ul local Colab, necesitând internet. Datele **nu** sunt incluse în proiect pentru a evita un pachet excesiv de mare.

In [ ]:
from cisco_integration import list_scenario_files, download_scenarios
for scene in SCENARIOS:
    items = list_scenario_files(scene)
    print('SCENARIO',scene, 'FILES:',[x.get('name') for x in items])
files = download_scenarios(SCENARIOS, directory='data/cisco_raw', max_file_mb=MAX_FILE_MB)
for key,path in files.items():
    assert Path(path).exists() and Path(path).stat().st_size>0
    print('VERIFIED',key,Path(path).name,round(Path(path).stat().st_size/1048576,2),'MB')

## 4b. Fișiere Cisco despre evenimente (pentru auditul etichetelor)

In [ ]:
from cisco_integration import download_case_images
from IPython.display import display, Image
try:
    case_images = download_case_images(SCENARIOS)
except Exception as err:
    case_images = {}
    print('Imaginile despre evenimente sunt opționale; download nereușit:',repr(err))
for scene,paths in case_images.items():
    print('CASE FILE – SCENARIO', scene)
    for path in paths[:2]:
        print(path.name)
        display(Image(filename=str(path),width=850))
print('Aceste imagini nu se transformă automat în ground truth.')

In [ ]:
from cisco_integration import load_cisco_timeseries
series, feature_cols = load_cisco_timeseries(
    files, resample_seconds=RESAMPLE_SECONDS,
    override_timestamp=TIME_COLUMN_OVERRIDES, max_features=MAX_FEATURES
)
print('\nFEATURES:',feature_cols)
for k,df in series.items():
    print(k, 'rows:',len(df), 'time:',df['__timestamp'].min(),'->',df['__timestamp'].max())
    display(df.head(3))

### Auditul caracteristicilor după agregare

In [ ]:
import pandas as pd
rows=[]
for scene, frame in series.items():
    for col in feature_cols:
        rows.append({
            'scenario':scene,'metric':col,
            'coverage_pct':round(100*frame[col].notna().mean(),1),
            'unique':int(frame[col].nunique(dropna=True)),
            'std':round(float(frame[col].std()),3)
        })
audit_features=pd.DataFrame(rows)
display(audit_features.pivot(index='metric',columns='scenario',values='unique'))
print('Metrici selectate:',len(feature_cols))
assert len(feature_cols)>=4, 'Au ramas prea putine metrici! Nu antrena modelele.'
assert any(c!='bandwidth' for c in feature_cols), 'Selected constant bandwidth again!'
assert any(audit_features.loc[audit_features.metric.eq(c), 'unique'].max()>1 for c in feature_cols), 'No variation'
print('PASS: multiple varying telemetry features across Cisco scenarios')

## 6. Etichete + split 70/15/15

In [ ]:
from cisco_integration import canonicalize_and_split
canonical_path, split_groups, split_audit = canonicalize_and_split(
    series,
    output='data/cisco_labeled.csv',
    mode=LABEL_MODE,
    events_csv=EVENTS_CSV,
    window=WINDOW,
    embargo=EMBARGO,
    split_ratios=(0.70,0.15,0.15)
)
print('LABEL MODE:',LABEL_MODE)
for sid,info in split_audit.items():
    print('Scenario',sid)
    for part,row in info.items():
        print(' ',part,'rows:',row['rows'],'normal:',row['normal'],
              'anomalous:',row['anomalous'],'time:',row['start'],'->',row['end'])
assert len(set(split_groups['train'])&set(split_groups['val'])) == 0
assert len(set(split_groups['val'])&set(split_groups['test'])) == 0

## 7. Construire `TelemetryWindowDataset` + `DataLoader` + transformări ajustate pe train

In [ ]:
from data_pipeline import prepare_data
bundle = prepare_data(canonical_path, splits=split_groups,
                      window=WINDOW, step=STEP, batch_size=BATCH_SIZE)
print(json.dumps(bundle.summary,indent=2))
assert set(bundle.datasets)=={'train','val','test'}
for part in ('train','val','test'):
    x,y=next(iter(bundle.loaders[part]))
    assert tuple(x.shape[1:3])==(1,WINDOW)
    assert x.shape[0] == y.shape[0]
    assert torch.isfinite(x).all()
    assert set(y.long().unique().tolist()).issubset({0,1})
    print('PASS',part,'X:',tuple(x.shape),'y:',tuple(y.shape),
          'normal:',bundle.summary[part]['normal'],
          'anormal:',bundle.summary[part]['anomalous'])
print('Dataset/DataLoader: PASS')

In [ ]:
import matplotlib.pyplot as plt
x0,y0 = bundle.datasets['train'][0]
plt.figure(figsize=(9,3))
plt.imshow(x0[0].numpy(),aspect='auto',cmap='viridis')
plt.colorbar(label='Z-score (clipped)')
plt.title(f'Fereastră 2D de telemetrie Cisco: label={int(y0)} ({LABEL_MODE})')
plt.xlabel('Caracteristici numerice (cu padding)');plt.ylabel('Pas temporal')
plt.tight_layout();plt.show()

## 8. Antrenare CNN + Convolutional Autoencoder, evaluare pe test

In [ ]:
if RUN_TRAINING:
    result_dir = 'results/cisco_'+LABEL_MODE
    cmd=[sys.executable,'src/train_compare.py','--data',str(canonical_path),
        '--train',','.join(split_groups['train']),
        '--val',','.join(split_groups['val']),
        '--test',','.join(split_groups['test']),
        '--window',str(WINDOW),'--step',str(STEP),
        '--batch-size',str(BATCH_SIZE),'--epochs',str(EPOCHS),
        '--output',result_dir]
    print('RUN:', ' '.join(cmd))
    status=subprocess.run(cmd, capture_output=True,text=True)
    print(status.stdout[-13000:])
    if status.stderr: print('STDERR:',status.stderr[-8000:])
    assert status.returncode==0, 'Antrenarea a eșuat; verifică logul afișat'
else:
    print('Antrenarea este dezactivată; Dataset/DataLoader sunt deja verificate.')

## 9. Rezultate și artefacte

In [ ]:
if RUN_TRAINING:
    from IPython.display import display, Image
    p=Path(result_dir)
    expected=['metrics.json','history.csv','confusion_matrices.png','loss_curves.png','cnn.pt','autoencoder.pt']
    for f in expected:
        assert (p/f).exists(),f'Missing result: {p/f}'
    report=json.loads((p/'metrics.json').read_text())
    print('MODEL SUMMARY –',LABEL_MODE)
    for model in ('CNN','Autoencoder'):
        print(model,report[model]['test'])
    display(Image(filename=str(p/'confusion_matrices.png')))
    display(Image(filename=str(p/'loss_curves.png')))
    if LABEL_MODE=='scenario_proxy':
        print('Aceste scoruri compară etichete-proxy, nu incidente Cisco etichetate temporal.')

In [ ]:
template=Path('events_intervals_TEMPLATE.csv')
if not template.exists(): template.write_text('scenario,start,end\n',encoding='utf-8')
print('Template evenimente:',template.resolve())
print('Mod curent:',LABEL_MODE)
if LABEL_MODE=='event_intervals':
    print('Ai selectat intervalele validate; verifică manual corespondenta cu case files Cisco')
else:
    print('Numai verificare tehnică. Pentru ground truth, completează template-ul cu timpi reali.')